# Query-Free Adversarial Attack on Stable Diffusion — reproduction and analysis
*Zhuang, Zhang & Liu, "A Pilot Study of Query-Free Adversarial Attack against Stable Diffusion" (CVPRW 2023, arXiv:2303.16378)*

**Idea of the paper.** Stable Diffusion (SD) sees a prompt only through its CLIP text encoder $\tau_\theta$. The attacker appends a
**5-character** string to the prompt so that the perturbed text embedding moves as far as possible from the original one:
$$\min_{x'}\ \cos\big(\tau_\theta(x),\ \tau_\theta(x')\big)$$
The attack never calls the diffusion model, hence **query-free**. A **targeted** variant removes one object (e.g. "a young man")
by computing this cosine only on **steerable key dimensions**. Those are the embedding dimensions whose difference
$\tau(s_i)-\tau(s_i')$ has the same sign across sentences with and without the target phrase:
$$I_j = \mathbb 1\Big[\big|\textstyle\sum_i \mathrm{sign}(d_{i,j})\big| > \epsilon n\Big],\qquad \min_{x'} \cos(\tau(x)\odot I,\ \tau(x')\odot I)$$

**This notebook**
1. Implements the four optimizers from the paper: **Random** baseline, **Greedy** character search, **Genetic** algorithm (50 generations × 20 candidates, mutation 0.3), and **PGD** on a simplex relaxation over single-character tokens (100 steps, lr 0.1).
2. Runs the untargeted and targeted attacks on SD v1.4, using the same seeds for clean and attacked prompts.
3. Evaluates with CLIP score, as in the paper's Table 1. It also uses a second, *independent* CLIP judge, because the attack optimizes the ViT-L/14 text encoder that the paper also uses for evaluation. It adds image-to-image similarity and a retention score for the targeted attack.
4. **Extra analyses:** *why* a suffix works (CLIP's causal mask: only the tail of the embedding changes), suffix vs. **prefix** placement, perturbation **budget** 1–10 characters, how well text-space cosine predicts image change, attack cost in encoder queries, and the relaxation gap in PGD.

**How to run**
- **Colab:** Runtime → **T4 GPU** → Run all.
- **Kaggle:** GPU accelerator, **Internet on** → Run all.
- Default settings: 12 untargeted + 6 targeted prompts × 5 seeds, about 45–55 min on a T4. `FAST = True` takes about 6 min. The paper's full scale is 20 prompts × 10 images.
- Results are saved to `qf_results/`.

*Scope note:* this is an academic robustness evaluation of a public model with benign prompts. SD's safety checker stays **on**, and any image it flags is excluded from scoring.

In [ ]:
# Colab preinstalls a torchao version that recent diffusers cannot import from ("cannot import name 'FqnToConfig'").
# torchao is not needed here, so remove it. If you already hit that error: run this cell, then Runtime -> Restart session.
%pip uninstall -y -q torchao
%pip install -q -U diffusers accelerate

In [ ]:
import os, time, string, math, json, warnings
import numpy as np, pandas as pd, torch, torch.nn.functional as F, matplotlib.pyplot as plt
from scipy import stats
from transformers import CLIPTokenizer, CLIPTextModel, CLIPModel, CLIPProcessor
from IPython.display import display
warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(no GPU: image generation will be very slow)")
OUT = "qf_results"; os.makedirs(f"{OUT}/images", exist_ok=True)

## 1. Configuration and prompts

In [ ]:
FAST = False
SD_CANDIDATES = ["CompVis/stable-diffusion-v1-4",                 # model used in the paper
                 "stable-diffusion-v1-5/stable-diffusion-v1-5"]   # fallback (same CLIP ViT-L/14 text encoder)
EMBED_MODE = "sequence"   # "sequence": full 77x768 hidden states that SD cross-attends to | "pooled": 768-d EOS embedding
N_CHARS    = 5
# CLIP's tokenizer lower-cases its input, so upper-case letters would only duplicate lower-case ones
CHARSET    = list(string.ascii_lowercase + string.digits + string.punctuation)
GA         = dict(generations=50, pop=20, mut=0.3)   # paper settings
PGD        = dict(steps=100, lr=0.1, n_samples=32, refine_topk=8, refine_rounds=3)   # paper: lr 0.1, 100 steps (+ discrete refinement)
PRESERVE_LAMBDA = 1.0                               # extension: weight of the "keep non-target dims unchanged" term
KEY_EPS    = 0.9                                     # paper: epsilon for key-dimension voting (n = 10 sentences)
SEEDS      = [0, 1] if FAST else list(range(5))      # paper uses 10 images per prompt (10 = ~1.5 h on a T4)
STEPS, GUIDANCE = 20, 7.5
RUN_PREFIX_IMAGES = not FAST                         # also generate images for the prefix-placement ablation
rng = np.random.default_rng(0); torch.manual_seed(0)

UNTARGETED_PROMPTS = [
    "A black bicycle against a brick wall", "A snake and a young man", "A cup of coffee on a wooden table",
    "A red apple on a white plate", "A dog running on the beach", "A lighthouse on a rocky coast at sunset",
    "A bowl of ramen with chopsticks", "An old man reading a newspaper on a bench", "A sailboat on a calm lake",
    "A cat sleeping on a sofa", "A snowy mountain under a blue sky", "A vase of sunflowers by the window",
    "A child flying a kite in a park", "A steam train crossing a bridge", "A teddy bear on a bed",
    "A horse grazing in a green field", "A woman holding an umbrella in the rain", "A plate of pancakes with strawberries",
    "A wooden cabin in a pine forest", "A yellow taxi on a city street at night"]
# (prompt, content to erase, content to keep)
TARGETED = [
    ("A black bicycle against a brick wall", "a black bicycle", "a brick wall"),
    ("A red apple on a white plate", "a white plate", "a red apple"),
    ("A snake and a young man", "a young man", "a snake"),
    ("A wooden cabin next to a lake", "a lake", "a wooden cabin"),
    ("A cat sitting in a basket", "a basket", "a cat"),
    ("A sailboat and a lighthouse", "a lighthouse", "a sailboat"),
    ("A dog and a red ball on the grass", "a red ball", "a dog on the grass"),
    ("A teddy bear and a lamp on a table", "a lamp", "a teddy bear on a table")]
# n = 10 neutral scenes used to find steerable key dimensions (paper used ChatGPT to write these)
SCENES = ["A bird flew high in the sky", "The sun set over the horizon", "A car drove down the road",
          "Leaves fell from the tree", "Waves crashed on the shore", "A train passed through the station",
          "Rain fell on the city", "A candle burned on the table", "Snow covered the rooftops", "Children played in the yard"]

UNTARGETED_PROMPTS = UNTARGETED_PROMPTS[: 4 if FAST else 12]
TARGETED = TARGETED[: 3 if FAST else 6]

## 2. The attacked model: SD's CLIP text encoder

The attack only needs SD's own tokenizer and text encoder, loaded here in fp32 for clean gradients. The diffusion model is loaded
later and used only to *evaluate* the attacks.

In [ ]:
SD_REPO = None
for repo in SD_CANDIDATES:
    try:
        tokenizer = CLIPTokenizer.from_pretrained(repo, subfolder="tokenizer")
        text_encoder = CLIPTextModel.from_pretrained(repo, subfolder="text_encoder").to(device).eval().requires_grad_(False)
        SD_REPO = repo; break
    except Exception as e:
        print(f"could not load {repo}: {type(e).__name__}: {str(e)[:150]}")
assert SD_REPO, "no Stable Diffusion repo could be loaded (check internet / HF access)"
L = tokenizer.model_max_length
print("attacking text encoder of", SD_REPO, "| max tokens", L, "| hidden", text_encoder.config.hidden_size)

def tokenize(prompts):
    return tokenizer(prompts, padding="max_length", max_length=L, truncation=True, return_tensors="pt").input_ids.to(device)

@torch.no_grad()
def encode(prompts, mode=None, bs=128):
    """tau_theta(x). 'sequence' = last_hidden_state (B,77,768), exactly what SD conditions on."""
    mode = mode or EMBED_MODE; outs = []
    for i in range(0, len(prompts), bs):
        o = text_encoder(tokenize(prompts[i:i + bs]))
        outs.append(o.last_hidden_state if mode == "sequence" else o.pooler_output)
    return torch.cat(outs)

def cos_to_ref(ref, embs, mask=None):
    a, b = ref.unsqueeze(0), embs
    if mask is not None:                       # key-dimension mask over the hidden dim (broadcast over positions)
        a, b = a * mask, b * mask
    return F.cosine_similarity(a.flatten(1), b.flatten(1), dim=-1)

def objective(ref, embs, mask=None, preserve=0.0):
    """Paper objective cos(tau(x)*I, tau(x')*I). With preserve > 0 (our extension), also reward keeping the
    NON-key dimensions unchanged, so a targeted attack erases the target without damaging the rest of the prompt."""
    c = cos_to_ref(ref, embs, mask)
    if preserve and mask is not None:
        c = c - preserve * cos_to_ref(ref, embs, 1 - mask)
    return c

def perturb(prompt, s, position="suffix"):
    return f"{prompt} {s}" if position == "suffix" else f"{s} {prompt}"

## 3. Attack implementations

All four methods minimise the same objective. Each counts how many text-encoder forward passes it uses, which is the attack's query
cost. None of them queries the diffusion model.

- **Random:** one random 5-character string (the paper's baseline).
- **Greedy:** build the string one character at a time, choosing at each step the character that lowers the cosine most. This is 5 × 68 candidates.
- **Genetic:** keep a population of 20 strings for 50 generations. Each generation keeps the top half (elitism), creates children by one-point crossover, and mutates each character with probability 0.3.
- **PGD:** relax each of the 5 slots to a point on the probability simplex over single-character CLIP tokens. The slot embedding is $W E$. Take normalised gradient steps on $W$, project back onto the simplex, then sample discrete strings from $W$ and keep the one with the lowest *true* loss. In our first run, rounding the soft solution to real characters undid most of the optimisation (relaxed cos 0.36 vs. real-string cos 0.77). We therefore add a short **discrete refinement**: a coordinate search over real strings in which each slot may only use its top-8 characters under $W$. The search is still guided by the gradient, and the notebook reports the cosine both before and after refinement. The real string is re-tokenized and re-encoded, so every reported number is for an actual prompt. A forward hook on the token-embedding layer injects the relaxed embeddings, which keeps the code independent of the `transformers` version.

In [ ]:
def make_evaluator(prompt, ref, mask=None, position="suffix", preserve=0.0):
    cache = {}
    def f(strings):
        new = [s for s in dict.fromkeys(strings) if s not in cache]
        if new:
            cache.update(zip(new, objective(ref, encode([perturb(prompt, s, position) for s in new]), mask, preserve).tolist()))
        return np.array([cache[s] for s in strings])
    f.cache = cache
    return f

def random_attack(f, k=N_CHARS):
    s = "".join(rng.choice(CHARSET, k)); f([s]); return s

def greedy_attack(f, k=N_CHARS):
    s = ""
    for _ in range(k):
        cands = [s + c for c in CHARSET]
        s = cands[int(f(cands).argmin())]
    return s

def genetic_attack(f, k=N_CHARS, generations=50, pop=20, mut=0.3):
    P = ["".join(rng.choice(CHARSET, k)) for _ in range(pop)]
    for _ in range(generations):
        c = f(P); order = np.argsort(c)
        elite = [P[i] for i in order[: pop // 2]]
        children = []
        while len(children) < pop - len(elite):
            a, b = rng.choice(len(elite), 2, replace=False)
            cut = int(rng.integers(0, k + 1))
            child = list(elite[a][:cut] + elite[b][cut:])
            for j in range(k):
                if rng.random() < mut:
                    child[j] = rng.choice(CHARSET)
            children.append("".join(child))
        P = elite + children
    c = f(P)
    return min(f.cache, key=f.cache.get) if len(f.cache) else P[int(c.argmin())]

# ---- PGD on a simplex relaxation over single-character tokens ----
vocab = tokenizer.get_vocab(); inv_vocab = {i: t for t, i in vocab.items()}
CAND = sorted(i for t, i in vocab.items() if len(t.replace("</w>", "")) == 1 and t.replace("</w>", "") in CHARSET)
emb_layer = text_encoder.get_input_embeddings()

class _Inject:                        # forward hook: overwrite token embeddings at given positions
    vec, pos = None, None
    def __call__(self, module, inputs, out):
        if self.vec is None: return out
        out = out.clone(); out[:, self.pos, :] = self.vec.to(out.dtype); return out
INJECT = _Inject(); emb_layer.register_forward_hook(INJECT)

def project_simplex(v):
    u, _ = torch.sort(v, descending=True, dim=1)
    css = u.cumsum(1) - 1
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((u - css / idx) > 0).to(v.dtype).mul(idx).argmax(1, keepdim=True)
    theta = css.gather(1, rho) / (rho + 1).to(v.dtype)
    return (v - theta).clamp_min(0)

def pgd_attack(f, prompt, ref, mask=None, position="suffix", k=N_CHARS, steps=100, lr=0.1, n_samples=32,
               preserve=0.0, refine_topk=8, refine_rounds=3):
    mid = tokenizer(prompt).input_ids[1:-1]
    filler = [CAND[0]] * k
    ids = [tokenizer.bos_token_id] + (mid + filler if position == "suffix" else filler + mid) + [tokenizer.eos_token_id]
    pos = list(range(1 + len(mid), 1 + len(mid) + k)) if position == "suffix" else list(range(1, 1 + k))
    ids = torch.tensor([ids + [tokenizer.pad_token_id] * (L - len(ids))], device=device)
    E = emb_layer.weight[CAND].detach()
    W = torch.full((k, len(CAND)), 1.0 / len(CAND), device=device)
    INJECT.pos = pos
    try:
        for _ in range(steps):
            W.requires_grad_(True)
            INJECT.vec = (W @ E).unsqueeze(0)
            o = text_encoder(ids)
            loss = objective(ref, o.last_hidden_state if EMBED_MODE == "sequence" else o.pooler_output, mask, preserve).sum()
            g, = torch.autograd.grad(loss, W)
            with torch.no_grad():
                W = project_simplex(W - lr * g / (g.norm(dim=1, keepdim=True) + 1e-12))
        relaxed_loss = loss.item()
    finally:
        INJECT.vec = None
    # every candidate is a single printable-ASCII token ("x" or "x</w>"), which maps to the character itself
    to_str = lambda toks: "".join(inv_vocab[CAND[t]].replace("</w>", "") for t in toks)
    cands = [to_str(W.argmax(1).tolist())]
    cands += [to_str(row) for row in torch.multinomial(W.clamp_min(1e-12), n_samples, replacement=True).T.tolist()]
    c = f(cands); best = cands[int(c.argmin())]
    pgd_attack.last_relaxed, pgd_attack.last_unrefined = relaxed_loss, float(c.min())
    # Discrete refinement (closes the relaxation gap): coordinate search over REAL strings, but each slot may only
    # use its top-k characters under the optimised distribution W, so the search stays gradient-guided.
    slot_chars = [list(dict.fromkeys(inv_vocab[CAND[t]].replace("</w>", "") for t in row))
                  for row in W.topk(min(refine_topk, W.shape[1]), dim=1).indices.tolist()]
    for _ in range(refine_rounds):
        improved = False
        for j in range(k):
            trials = [best[:j] + ch + best[j + 1:] for ch in slot_chars[j]]
            tc = f(trials); i = int(tc.argmin())
            if tc[i] < f([best])[0] - 1e-6:
                best, improved = trials[i], True
        if not improved:
            break
    return best

print(f"{len(CHARSET)} characters | {len(CAND)} single-character CLIP tokens for PGD")

In [ ]:
METHODS = ["random", "greedy", "genetic", "pgd"]

def run_attack(method, prompt, ref, mask=None, position="suffix", k=N_CHARS, preserve=0.0):
    f = make_evaluator(prompt, ref, mask, position, preserve)
    t0 = time.time()
    if method == "random":    s = random_attack(f, k)
    elif method == "greedy":  s = greedy_attack(f, k)
    elif method == "genetic": s = genetic_attack(f, k, **GA)
    elif method == "pgd":     s = pgd_attack(f, prompt, ref, mask, position, k, preserve=preserve, **PGD)
    queries = len(f.cache) + (PGD["steps"] if method == "pgd" else 0)   # PGD: + forward/backward passes
    return dict(suffix=s, adv_prompt=perturb(prompt, s, position), cos=float(f([s])[0]),
                seconds=time.time() - t0, encoder_queries=queries,
                relaxed_cos=getattr(pgd_attack, "last_relaxed", np.nan) if method == "pgd" else np.nan,
                unrefined_cos=getattr(pgd_attack, "last_unrefined", np.nan) if method == "pgd" else np.nan)

# ---------------- untargeted ----------------
U = []
for pi, prompt in enumerate(UNTARGETED_PROMPTS):
    ref = encode([prompt])[0]
    for m in METHODS:
        r = run_attack(m, prompt, ref); U.append(dict(prompt_id=pi, prompt=prompt, method=m, position="suffix", **r))
    for m in ["greedy", "genetic"]:                                          # placement ablation
        r = run_attack(m, prompt, ref, position="prefix"); U.append(dict(prompt_id=pi, prompt=prompt, method=m, position="prefix", **r))
    print(f"[{pi}] {prompt!r}: " + ", ".join(f"{x['method']}{'/pre' if x['position'] == 'prefix' else ''}={x['suffix']!r}({x['cos']:.3f})" for x in U if x["prompt_id"] == pi))
U = pd.DataFrame(U); U["condition"] = np.where(U.position == "suffix", U.method, U.method + "_prefix")
U.to_csv(f"{OUT}/untargeted_attacks.csv", index=False)

### Targeted attack: steerable key dimensions

For each target phrase *t* we build $s_i$ = "{scene$_i$} and *t*" and $s_i'$ = "{scene$_i$}", where $i = 1..10$. We embed both with the EOS
(pooled) representation and keep the dimensions where all 10 differences agree in sign ($\epsilon = 0.9 \Rightarrow |\sum \mathrm{sign}| > 9$).
In `sequence` mode the 768-d mask is applied at every token position.

Besides the paper's four methods (suffix), two extra conditions are run with the Genetic optimiser:
- **`genetic_prefix`:** the same masked objective, with the perturbation placed *before* the prompt.
- **`genetic_preserve`** *(our extension)*: the first run showed the paper's masked objective also damages the content that should be kept. This variant minimises $\cos(\tau(x)\odot I,\tau(x')\odot I) - \lambda\cos(\tau(x)\odot(1-I),\tau(x')\odot(1-I))$ with $\lambda=1$. It still erases along the key dimensions, but also explicitly rewards leaving all other dimensions unchanged.

In [ ]:
def key_dim_mask(target, eps=KEY_EPS):
    s  = [f"{sc} and {target}" for sc in SCENES]
    d  = encode(s, mode="pooled") - encode(SCENES, mode="pooled")
    vote = torch.sign(d).sum(0).abs()
    I = (vote > eps * len(SCENES)).float()
    if I.sum() < 10:   # safeguard: fall back to the 50 strongest dimensions
        I = torch.zeros_like(vote); I[torch.topk(vote + d.abs().mean(0) * 1e-3, min(50, len(vote))).indices] = 1
    return I

TARGETED_CONDS = [("random", "suffix", 0.0), ("greedy", "suffix", 0.0), ("genetic", "suffix", 0.0), ("pgd", "suffix", 0.0),
                  ("genetic", "prefix", 0.0), ("genetic", "suffix", PRESERVE_LAMBDA)]
cond_name = lambda m, pos, pres: m + ("_prefix" if pos == "prefix" else "") + ("_preserve" if pres else "")

Tg = []
for ti, (prompt, target, keep) in enumerate(TARGETED):
    ref, I = encode([prompt])[0], key_dim_mask(target)
    for m, pos, pres in TARGETED_CONDS:
        r = run_attack(m, prompt, ref, mask=I, position=pos, preserve=pres)
        e = encode([r["adv_prompt"]])
        r.update(cos_full=float(cos_to_ref(ref, e)[0]), cos_masked=float(cos_to_ref(ref, e, I)[0]),
                 cos_complement=float(cos_to_ref(ref, e, 1 - I)[0]))
        Tg.append(dict(prompt_id=ti, prompt=prompt, target=target, keep=keep, method=m, position=pos,
                       condition=cond_name(m, pos, pres), key_dims=int(I.sum()), **r))
    print(f"[{ti}] erase {target!r} ({int(I.sum())} key dims): " + ", ".join(f"{x['condition']}={x['suffix']!r}" for x in Tg if x["prompt_id"] == ti))
Tg = pd.DataFrame(Tg); Tg.to_csv(f"{OUT}/targeted_attacks.csv", index=False)

In [ ]:
# Text-space results: lower cosine = stronger attack on the encoder (no images involved yet)
txt_u = U.groupby(["position", "method"]).agg(cos_mean=("cos", "mean"), cos_std=("cos", "std"), seconds=("seconds", "mean"),
                                             encoder_queries=("encoder_queries", "mean")).reset_index()
txt_t = Tg.groupby("condition").agg(masked_cos=("cos_masked", "mean"), non_key_cos=("cos_complement", "mean"),
                                    full_cos=("cos_full", "mean"), key_dims=("key_dims", "mean"), seconds=("seconds", "mean")).reset_index()
print("Untargeted (text space):"); display(txt_u.round(4))
print("Targeted (text space): selective erasure = low masked_cos with high non_key_cos"); display(txt_t.round(4))
pg = U[(U.method == "pgd")]
print(f"PGD: relaxed cos {pg.relaxed_cos.mean():.3f} | best sampled string {pg.unrefined_cos.mean():.3f} "
      f"| after discrete refinement {pg.cos.mean():.3f}")

## 4. Why does a 5-character suffix work? CLIP's causal mask

CLIP's text transformer uses a **causal** attention mask, so a token's hidden state depends only on the tokens before it. Appending a
suffix therefore leaves the hidden states of the original prompt tokens **exactly unchanged**. Only the suffix tokens, the EOS token,
and the ~60 EOS *padding* positions change. SD cross-attends to all 77 positions, and the padding positions summarise the whole
prompt, so the attack works by corrupting that tail. A **prefix** works differently: it changes every position. The next cell checks this.

In [ ]:
pi = 0; prompt = UNTARGETED_PROMPTS[pi]
suf = U[(U.prompt_id == pi) & (U.method == "genetic") & (U.position == "suffix")].adv_prompt.iloc[0]
pre = U[(U.prompt_id == pi) & (U.method == "genetic") & (U.position == "prefix")].adv_prompt.iloc[0]
H = encode([prompt, suf, pre], mode="sequence")
n_prompt_tok = len(tokenizer(prompt).input_ids) - 1           # BOS + words (EOS excluded)
per_pos = {name: F.cosine_similarity(H[0], H[j], dim=-1).cpu().numpy() for j, name in [(1, f"suffix: {suf!r}"), (2, f"prefix: {pre!r}")]}
fig, ax = plt.subplots(figsize=(10, 3.5))
for name, v in per_pos.items(): ax.plot(v, marker=".", label=name)
ax.axvspan(0, n_prompt_tok - 0.5, color="green", alpha=.08, label="original prompt tokens")
ax.set_xlabel("token position (0 = BOS, rest = EOS padding)"); ax.set_ylabel("cos(original, attacked)"); ax.legend(fontsize=8)
ax.set_title("Per-position hidden-state similarity"); plt.tight_layout(); plt.savefig(f"{OUT}/fig_causal_positions.png", dpi=150); plt.show()
v = list(per_pos.values())
print(f"suffix attack: max |1-cos| on original-prompt positions = {np.abs(1 - v[0][:n_prompt_tok]).max():.2e} (exactly unchanged)")
print(f"suffix attack: mean cos on padding tail = {v[0][n_prompt_tok + 3:].mean():.3f} | prefix attack: mean cos on prompt positions = {v[1][:n_prompt_tok].mean():.3f}")

In [ ]:
# Perturbation-budget ablation (text space only): greedy attack with k = 1..10 characters, suffix vs prefix
budget_rows = []
for pi, prompt in enumerate(UNTARGETED_PROMPTS[:6]):
    ref = encode([prompt])[0]
    for position in ["suffix", "prefix"]:
        f = make_evaluator(prompt, ref, position=position)
        s = ""
        for k in range(1, 11):                       # greedy is incremental, so one pass gives every budget
            cands = [s + c for c in CHARSET]; s = cands[int(f(cands).argmin())]
            budget_rows.append(dict(prompt_id=pi, position=position, k=k, cos=float(f([s])[0])))
budget = pd.DataFrame(budget_rows)
b = budget.groupby(["position", "k"]).cos.mean().unstack(0)
b.plot(marker="o", figsize=(6, 3.5)); plt.ylabel("cos to original (lower = stronger)"); plt.xlabel("characters added (greedy)")
plt.axvline(5, ls="--", c="gray"); plt.title("Perturbation budget"); plt.tight_layout(); plt.savefig(f"{OUT}/fig_budget.png", dpi=150); plt.show()
display(b.round(3).T)

## 5. Generating images with Stable Diffusion

Every prompt (clean and attacked) uses the same seeds, so differences come from the text only. The scheduler is DPM-Solver++ with 20
steps (the paper doesn't specify), guidance 7.5, 512×512.

In [ ]:
try:
    from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
except (ImportError, RuntimeError) as e:
    raise RuntimeError("diffusers failed to import (usually the torchao conflict). Run the install cell at the top, "
                       "then Runtime -> Restart session and Run all again.") from e
dtype = torch.float16 if device == "cuda" else torch.float32
pipe = StableDiffusionPipeline.from_pretrained(SD_REPO, torch_dtype=dtype).to(device)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)
IMAGES = {}            # (experiment, prompt_id, condition) -> list of PIL images (one per seed)
NSFW_FLAGS = {}

@torch.no_grad()
def generate(key, prompt):
    gens = [torch.Generator(device).manual_seed(s) for s in SEEDS]
    out = pipe([prompt] * len(SEEDS), generator=gens, num_inference_steps=STEPS, guidance_scale=GUIDANCE)
    IMAGES[key] = out.images
    NSFW_FLAGS[key] = list(out.nsfw_content_detected or [False] * len(SEEDS))
    for s, im in zip(SEEDS, out.images):
        im.save(f"{OUT}/images/{key[0]}_{key[1]:02d}_{key[2]}_seed{s}.png")

t0 = time.time()
for pi, prompt in enumerate(UNTARGETED_PROMPTS):
    generate(("untargeted", pi, "none"), prompt)
    for _, r in U[(U.prompt_id == pi) & ((U.position == "suffix") | RUN_PREFIX_IMAGES)].iterrows():
        generate(("untargeted", pi, r.condition), r.adv_prompt)
    print(f"untargeted {pi + 1}/{len(UNTARGETED_PROMPTS)}  {time.time() - t0:.0f}s")
for ti, (prompt, target, keep) in enumerate(TARGETED):
    generate(("targeted", ti, "none"), prompt)
    for _, r in Tg[Tg.prompt_id == ti].iterrows():
        generate(("targeted", ti, r.condition), r.adv_prompt)
    print(f"targeted {ti + 1}/{len(TARGETED)}  {time.time() - t0:.0f}s")
print(f"{sum(len(v) for v in IMAGES.values())} images, {sum(sum(v) for v in NSFW_FLAGS.values())} flagged by the safety checker (excluded)")
del pipe; torch.cuda.empty_cache()

## 6. Evaluation

- **CLIP score** = cosine(image embedding, text embedding), the same metric as the paper's Table 1.
    - *Untargeted:* text = the **original** prompt. Lower means the image no longer matches what the user asked for.
    - *Targeted:* text = "This is a photo of {target}" (lower = object erased) and "This is a photo of {kept content}" (higher = rest preserved).
- **Two judges:** `openai/clip-vit-large-patch14` (the paper's choice; its text tower is the attacked encoder) and `openai/clip-vit-base-patch32`
  (an independent model). If the second judge agrees, the drop is not an artefact of evaluating with the attacked model.
- **Image–image similarity** to the clean image with the same seed: how much the picture itself changed.

In [ ]:
class ClipJudge:
    def __init__(self, name):
        self.name = name.split("/")[-1]
        self.model = CLIPModel.from_pretrained(name).to(device).eval()
        self.proc = CLIPProcessor.from_pretrained(name)
    @torch.no_grad()
    def img(self, images):
        px = self.proc(images=images, return_tensors="pt")["pixel_values"].to(device)
        return F.normalize(self.model.visual_projection(self.model.vision_model(pixel_values=px).pooler_output), dim=-1)
    @torch.no_grad()
    def txt(self, texts):
        t = self.proc.tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
        return F.normalize(self.model.text_projection(self.model.text_model(**t).pooler_output), dim=-1)

JUDGES = [ClipJudge("openai/clip-vit-large-patch14"), ClipJudge("openai/clip-vit-base-patch32")]
rows = []
for (exp, pid, cond), ims in IMAGES.items():
    ok = [not f for f in NSFW_FLAGS[(exp, pid, cond)]]
    for J in JUDGES:
        E = J.img(ims); E0 = J.img(IMAGES[(exp, pid, "none")])
        if exp == "untargeted":
            texts = {"score": UNTARGETED_PROMPTS[pid]}
        else:
            _, target, keep = TARGETED[pid]
            texts = {"score": f"This is a photo of {target}", "keep_score": f"This is a photo of {keep}"}
        T = {k: J.txt([v])[0] for k, v in texts.items()}
        for si, seed in enumerate(SEEDS):
            if not ok[si]: continue
            r = dict(exp=exp, prompt_id=pid, condition=cond, seed=seed, judge=J.name,
                      img_sim_to_clean=float(E[si] @ E0[si]), **{k: float(E[si] @ v) for k, v in T.items()})
            rows.append(r)
scores = pd.DataFrame(rows); scores.to_csv(f"{OUT}/clip_scores.csv", index=False)

order = ["none", "random", "greedy", "genetic", "pgd", "greedy_prefix", "genetic_prefix", "genetic_preserve"]
def table(exp, cols):
    s = scores[scores.exp == exp]
    t = s.groupby(["judge", "condition"])[cols].agg(["mean", "std"])
    t.columns = [f"{a}_{b}" for a, b in t.columns]; t = t.reset_index()
    t["condition"] = pd.Categorical(t.condition, [c for c in order if c in t.condition.unique()]); return t.sort_values(["judge", "condition"])
T1u = table("untargeted", ["score", "img_sim_to_clean"]); T1t = table("targeted", ["score", "keep_score", "img_sim_to_clean"])
T1u.to_csv(f"{OUT}/table_untargeted.csv", index=False); T1t.to_csv(f"{OUT}/table_targeted.csv", index=False)
print("UNTARGETED: CLIP score vs. the original prompt (paper Table 1, top row)"); display(T1u.round(3))
print("TARGETED: score = target object (lower is better), keep_score = remaining content (higher is better)"); display(T1t.round(3))

In [ ]:
# Statistical check: paired over prompts (seed-averaged), each attack vs the Random baseline and vs No attack (Wilcoxon signed-rank)
def paired_tests(exp, judge):
    s = scores[(scores.exp == exp) & (scores.judge == judge)].groupby(["prompt_id", "condition"]).score.mean().unstack()
    out = []
    for m in [c for c in order if c in s.columns and c != "none"]:
        for base in ["none", "random"]:
            if m == base or base not in s: continue
            d = (s[m] - s[base]).dropna()
            p = stats.wilcoxon(d).pvalue if len(d) >= 5 and np.any(d != 0) else np.nan
            out.append(dict(exp=exp, judge=judge, attack=m, vs=base, mean_diff=d.mean(), n_prompts=len(d), wilcoxon_p=p))
    return out
tests = pd.DataFrame(sum([paired_tests(e, j.name) for e in ["untargeted", "targeted"] for j in JUDGES], []))
tests.to_csv(f"{OUT}/paired_tests.csv", index=False); display(tests.round(4))

### Targeted attack: is the object actually erased?

An average CLIP score can't tell "erased" from "never drawn". In the first run, for example, SD often left out "a young man" even
*without* an attack. We therefore detect objects per image with a **calibrated threshold**. For each phrase, we score it against clean
images of *unrelated* prompts (ones that don't mention the object) and take their 95th percentile as the threshold. An image above
it counts as showing the object, with a ~5% false-positive rate.

- **Eligible:** a (prompt, seed) pair is eligible only if the target is present in the *clean* image with that seed.
- **erase_rate:** the fraction of eligible pairs where the target disappears.
- **clean_erase_rate:** the fraction where the target disappears *and* the kept content is still detected (selective erasure).

In [ ]:
pres_rows, prompt_rows = [], []
for J in JUDGES:
    thr = {}
    for ti, (prompt, target, keep) in enumerate(TARGETED):
        for kind, phrase in (("target", target), ("keep", keep)):
            noun = phrase.split()[-1].lower()
            neg = [im for pi, p in enumerate(UNTARGETED_PROMPTS) if noun not in p.lower()
                   for im, fl in zip(IMAGES[("untargeted", pi, "none")], NSFW_FLAGS[("untargeted", pi, "none")]) if not fl]
            sc = (J.img(neg) @ J.txt([f"This is a photo of {phrase}"])[0]).float().cpu().numpy()
            thr[(ti, kind)] = float(np.quantile(sc, 0.95))
    sj = scores[(scores.exp == "targeted") & (scores.judge == J.name)].copy()
    sj["t"] = [r.score > thr[(r.prompt_id, "target")] for r in sj.itertuples()]
    sj["k"] = [r.keep_score > thr[(r.prompt_id, "keep")] for r in sj.itertuples()]
    mg = sj.merge(sj[sj.condition == "none"][["prompt_id", "seed", "t", "k"]].rename(columns={"t": "t0", "k": "k0"}),
                  on=["prompt_id", "seed"])
    for cond, g in mg.groupby("condition"):
        el = g[g.t0]
        pres_rows.append(dict(judge=J.name, condition=cond, target_present=g.t.mean(), keep_present=g.k.mean(),
                              n_eligible=len(el), erase_rate=(~el.t).mean() if len(el) else np.nan,
                              clean_erase_rate=((~el.t) & el.k).mean() if len(el) else np.nan))
    for ti, g in mg[mg.condition == "none"].groupby("prompt_id"):
        prompt_rows.append(dict(judge=J.name, prompt=TARGETED[ti][0], target=TARGETED[ti][1],
                                clean_target_present=g.t.mean(), clean_keep_present=g.k.mean()))
presence = pd.DataFrame(pres_rows)
presence["condition"] = pd.Categorical(presence.condition, [c for c in order if c in presence.condition.unique()])
presence = presence.sort_values(["judge", "condition"]); presence.to_csv(f"{OUT}/targeted_presence.csv", index=False)
clean_presence = pd.DataFrame(prompt_rows)
print("Does the CLEAN model draw both objects? (prompts with low clean_target_present cannot be erased)")
display(clean_presence.round(2))
print("Erasure results (target_present: lower = better; keep_present / clean_erase_rate: higher = better)")
display(presence.round(3))

In [ ]:
# Does text-space damage predict image-space damage? (per prompt x method, seed-averaged, ViT-L judge)
s = scores[(scores.exp == "untargeted") & (scores.judge == JUDGES[0].name)].groupby(["prompt_id", "condition"]).score.mean().reset_index()
u = U[["prompt_id", "condition", "cos"]]
none_cos = pd.DataFrame(dict(prompt_id=range(len(UNTARGETED_PROMPTS)), condition="none", cos=1.0))
mrg = s.merge(pd.concat([u, none_cos]), on=["prompt_id", "condition"])
rho, p = stats.spearmanr(mrg.cos, mrg.score)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for c, g in mrg.groupby("condition"): axes[0].scatter(g.cos, g.score, label=c, s=25)
axes[0].set_xlabel("text-embedding cos to original (attack objective)"); axes[0].set_ylabel("CLIP score vs original prompt")
axes[0].set_title(f"Spearman rho = {rho:.2f} (p = {p:.1e})"); axes[0].legend(fontsize=8)
sv = scores[scores.exp == "untargeted"].groupby(["judge", "condition"]).score.mean().unstack(0)
sv = sv.loc[[c for c in order if c in sv.index]]; sv.plot.bar(ax=axes[1], rot=30); axes[1].set_ylabel("mean CLIP score"); axes[1].set_title("Untargeted, both judges")
axes[1].set_ylim(sv.values.min() * 0.85, sv.values.max() * 1.05)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_text_vs_image.png", dpi=150); plt.show()

In [ ]:
# Qualitative grids (paper Fig. 3 / Fig. 4 analogues)
def grid(exp, ids, conds, fname, title_fn):
    rows_ = [(i, c) for i in ids for c in conds if (exp, i, c) in IMAGES]
    fig, axes = plt.subplots(len(rows_), len(SEEDS), figsize=(2.6 * len(SEEDS), 2.75 * len(rows_)), squeeze=False)
    for r, (i, c) in enumerate(rows_):
        for k in range(len(SEEDS)):
            axes[r, k].imshow(IMAGES[(exp, i, c)][k]); axes[r, k].axis("off")
        axes[r, 0].set_title(title_fn(i, c), fontsize=8, loc="left")
    plt.tight_layout(); plt.savefig(f"{OUT}/{fname}", dpi=110); plt.show()

def adv_text(df, i, c):
    if c == "none": return None
    return df[(df.prompt_id == i) & (df.condition == c)].adv_prompt.iloc[0]

grid("untargeted", range(min(4, len(UNTARGETED_PROMPTS))), ["none", "genetic", "pgd"], "fig_grid_untargeted.png",
     lambda i, c: f"[{c}] " + (adv_text(U, i, c) or UNTARGETED_PROMPTS[i]))
grid("targeted", range(len(TARGETED)), ["none", "genetic", "genetic_prefix", "genetic_preserve"], "fig_grid_targeted.png",
     lambda i, c: f"[{c}, erase '{TARGETED[i][1]}'] " + (adv_text(Tg, i, c) or TARGETED[i][0]))

In [ ]:
# What did the attacks find? Token view of a few suffixes (cf. the paper's 'WALL-E' case study)
for _, r in U[(U.method.isin(["greedy", "genetic", "pgd"])) & (U.position == "suffix")].head(12).iterrows():
    print(f"{r.method:8s} {r.prompt!r:45s} + {r.suffix!r:9s} -> tokens {tokenizer.tokenize(r.suffix)}  cos={r.cos:.3f}")

## 7. Draft report tables

`qf_results/report_draft.md` collects the numbers. Write the interpretation yourself from the tables and figures. The questions at the
end of the file are prompts for that.

In [ ]:
def md_table(d):
    try: return d.to_markdown(index=False, floatfmt=".3f")
    except Exception: return "```\n" + d.round(3).to_string(index=False) + "\n```"
parts = [f"# Query-free attack on {SD_REPO} ({len(UNTARGETED_PROMPTS)} untargeted + {len(TARGETED)} targeted prompts, {len(SEEDS)} seeds, {STEPS} steps)\n",
         f"Objective on `{EMBED_MODE}` embedding; {N_CHARS}-char perturbations from {len(CHARSET)} characters.\n",
         "## Text-space attack strength (untargeted)\n", md_table(txt_u),
         "\n## Text-space (targeted)\n", md_table(txt_t),
         "\n## Untargeted CLIP scores\n", md_table(T1u), "\n## Targeted CLIP scores\n", md_table(T1t),
         "\n## Targeted: object presence / erasure\n", md_table(presence), "\n", md_table(clean_presence),
         "\n## Paired tests\n", md_table(tests),
         f"\n## Text cos vs CLIP score\nSpearman rho = {rho:.3f} (p = {p:.2e})\n",
         "\n## Budget ablation (mean cos)\n", md_table(b.reset_index()),
         "\n## Figures\nfig_causal_positions.png, fig_budget.png, fig_text_vs_image.png, fig_grid_untargeted.png, fig_grid_targeted.png\n",
         "\n## Questions to answer in the write-up\n",
         "- Which optimizer is strongest in text space and in image space, and at what query cost? Do you reproduce Genetic > PGD > Greedy > Random?\n",
         "- Is the CLIP-score drop significant (paired tests) and does the independent ViT-B/32 judge agree?\n",
         "- What does the causal-mask plot imply about *why* suffix attacks work, and why prefixes are stronger?\n",
         "- Targeted: is the target removed while the rest is kept (erase_rate, clean_erase_rate)? Does the preserve term (our extension) or prefix placement help?\n",
         "- How well does text-embedding cosine predict image change? How large was PGD's relaxation gap and how much did discrete refinement recover?\n",
         "- Defences / limitations: perplexity or character-pattern filtering of prompts, robust text encoders, small sample size, CLIP score as a proxy.\n"]
open(f"{OUT}/report_draft.md", "w").write("\n".join(parts))
import shutil; shutil.make_archive("qf_results", "zip", OUT)
print(open(f"{OUT}/report_draft.md").read()[:3000]); print("\nsaved qf_results.zip")